In [1]:
# -*- coding: utf-8 -*-
import os, glob
import pandas as pd
from snownlp import SnowNLP

# 路径
r1 = r'D:/NJU_WORKPIECE/Project/每期弹幕/每期弹幕'
r2 = r'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP均值.xlsx'

records = []

# 遍历 1-349 期
for phase in range(1, 11):
    phase_dir = os.path.join(r1, f'第{phase}期')
    if not os.path.isdir(phase_dir):
        continue

    for fp in glob.glob(os.path.join(phase_dir, '*.txt')):
        with open(fp, 'r', encoding='utf-8') as f:
            danmu_list = [line.strip() for line in f if line.strip()]
        if not danmu_list:
            continue

        # SnowNLP 打分 0~1
        scores = [SnowNLP(d).sentiments for d in danmu_list]
        mean_score = round(sum(scores) / len(scores), 4)

        records.append({
            '期号': phase,
            '文件名': os.path.basename(fp),
            'SnowNLP均值': mean_score
        })

# 导出
df_out = pd.DataFrame(records)
df_out.to_excel(r2, index=False)
print('SnowNLP 均值结果已保存至', r2)

SnowNLP 均值结果已保存至 D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP均值.xlsx


In [7]:
import os, glob
import pandas as pd
from snownlp import SnowNLP

In [8]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

# 1. 读取数据
file_path = r"D:/NJU_WORKPIECE/Project/人工评分/人工评分.xlsx"
df = pd.read_excel(file_path)

# 2. 取前3000条作为训练集，其余为测试集
df_train = df.iloc[:3000].copy()
df_test  = df.iloc[3000:].copy()
df_train

,弹幕内容,情绪评分（人工填写）,情绪评分
0,银板和约,3,0.50
1,有些时候你从不会知道一个时刻的珍贵，直到它变成回忆,4,0.75
2,椰子牌的水,3,0.50
3,我是礼貌的魔鬼！！！！！已经走火入了魔！,5,1.00
4,固若金汤的防守,3,0.50
...,...,...,...
2995,真是令人摸不着头,3,0.50
2996,太对了，就是这个感觉,5,1.00
2997,?街健对握力要求极高，常年练小臂力量自然爆炸,3,0.50
2998,这个手震撼我了,5,1.00


In [9]:
df_test.head()  #测试集

,弹幕内容,情绪评分（人工填写）,情绪评分
3000,刘仪伟,3,0.5
3001,'坑洞郎朗进化后是新的植物,3,0.5
3002,这个鲤鱼,3,0.5
3003,懂了，是律师,3,0.5
3004,勃勃夫,3,0.5


In [17]:
dfpos=df_train[df_train["情绪评分（人工填写）"]>3]
dfneg=df_train[df_train["情绪评分（人工填写）"]<3]
dfpos

,弹幕内容,情绪评分（人工填写）,情绪评分
1,有些时候你从不会知道一个时刻的珍贵，直到它变成回忆,4,0.75
3,我是礼貌的魔鬼！！！！！已经走火入了魔！,5,1.00
9,细节左上角三连,4,0.75
10,没官配，可以刷,4,0.75
11,明天去买票,4,0.75
...,...,...,...
2991,这个妹妹好看,5,1.00
2992,背景好评,4,0.75
2994,哈哈哈哈哈，花絮就逗比了,5,1.00
2996,太对了，就是这个感觉,5,1.00


In [18]:
low_output_path = r'D:/NJU_WORKPIECE/Project/NEG.txt'  #积极情绪储存的文件
high_output_path = r'D:/NJU_WORKPIECE/Project/POS.txt'  #消极情绪储存的文件

def save_txt(df, output_path):
    # 提取 'txt' 列并过滤掉任何空值（NaN）
    txt_data = df['弹幕内容'].dropna().astype(str)
    # 将每行数据用换行符分隔并合并成一个字符串
    text_data = '\n'.join(txt_data)
    # 写入文本文件
    with open(output_path, 'w', encoding='utf-8') as file:
        file.write(text_data)
    print(f"文本已成功写入 {output_path}")

# 分别对 dfneg 和 dfpos 进行操作
save_txt(dfneg, low_output_path)
save_txt(dfpos, high_output_path)

文本已成功写入 D:/NJU_WORKPIECE/Project/NEG.txt
文本已成功写入 D:/NJU_WORKPIECE/Project/POS.txt


In [19]:
from snownlp import sentiment 
sentiment.train(r'D:/NJU_WORKPIECE/Project/NEG.txt',r'D:/NJU_WORKPIECE/Project/POS.txt')
sentiment.save(r'D:/NJU_WORKPIECE/Project/train.marshal')
sentiment.load(r'D:/NJU_WORKPIECE/Project/train.marshal')

In [10]:
right=0
wrong=0
test_df=df_test
for i in test_df.index:
    txt=test_df.loc[i,'弹幕内容']
    ren=test_df.loc[i,'情绪评分（人工填写）']
    test_df.loc[i,'NLP']=SnowNLP(txt).sentiments
    nlp=test_df.loc[i,'NLP']
    #print((ren,nlp))
    if  ren>=3 and nlp>=0.5 or ren<=3 and nlp<=0.5:
        right += 1
    else:
        wrong += 1
print("机器评分正确率为：",right/(right+wrong))

机器评分正确率为： 0.7965970610982211


In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. 取出人工标签与 SnowNLP 概率值
y_true = test_df['情绪评分（人工填写）'].values          # 1-5 整数
y_prob = test_df['NLP'].values                           # 0-1 浮点

# 2. 统一量纲：把人工 1-5 映射到 0-1，方便与 SnowNLP 概率比
y_true_01 = (y_true - 1) / 4

# 3. 计算回归指标
mse  = mean_squared_error(y_true_01, y_prob)
mae  = mean_absolute_error(y_true_01, y_prob)
r2   = r2_score(y_true_01, y_prob)

print(f'MSE : {mse:.4f}')
print(f'MAE : {mae:.4f}')
print(f'R²  : {r2:.4f}')

MSE : 0.1186
MAE : 0.2655
R²  : -0.4332


In [27]:
import os
import re
import pandas as pd
from snownlp import SnowNLP
from snownlp import sentiment

# 0. 载入自定义模型
sentiment_path = r'D:/NJU_WORKPIECE/Project/train.marshal'
sentiment.load(sentiment_path)

# 1. 根目录
root_dir = r'D:/NJU_WORKPIECE/Project/每期弹幕/每期弹幕301-349'

# 2. 结果容器
records = []

# 3. 递归遍历
for dirpath, _, filenames in os.walk(root_dir):
    for fn in filenames:
        if not fn.lower().endswith('.txt'):
            continue
        txt_path = os.path.join(dirpath, fn)

        # 抠期号
        folder_name = os.path.basename(dirpath)
        m = re.search(r'第(\d+)期', folder_name)
        if not m:
            continue
        week = int(m.group(1))

        # 4. 逐行读 + 逐行打分 + 求均值
        score_sum = 0.0
        line_cnt  = 0
        try:
            with open(txt_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    score_sum += SnowNLP(line).sentiments
                    line_cnt  += 1
        except Exception as e:
            print('读取出错，跳过：', txt_path, e)
            continue
        if line_cnt == 0:          # 空文件跳过
            continue
        avg_score = score_sum / line_cnt

        # 5. 记录结果（只留均值）
        records.append({
            '期号': week,
            '文件名': fn,
            '行数': line_cnt,
            'SnowNLP均值': round(avg_score, 3)
        })

# 6. 生成 DataFrame 并排序
df_out = pd.DataFrame(records)
df_out = df_out.sort_values('期号').reset_index(drop=True)

# 7. 导出
out_xlsx = r'D:/NJU_WORKPIECE/Project/01/各期弹幕_Training_SnowNLP均值301-349.xlsx'
df_out.to_excel(out_xlsx, index=False)
print(f'全部完成！结果已保存至：{out_xlsx}')

全部完成！结果已保存至：D:/NJU_WORKPIECE/Project/01/各期弹幕_Training_SnowNLP均值301-349.xlsx


In [30]:
import pandas as pd
import re

# 1. 读文件
df = pd.read_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分.xlsx', engine='openpyxl')

# 2. 提取长数字（≥6 位，可自己改）
df['cid'] = df['文件名'].astype(str).str.extract(r'(\d{6,})')

# 3. 保存
out_path = r'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_带CID.xlsx'
df.to_excel(out_path, index=False)

print('已生成：', out_path)

已生成： D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_带CID.xlsx


In [31]:
import pandas as pd

# 读取文件
csv_df = pd.read_csv('D:/NJU_WORKPIECE/Project/每周必看视频信息.csv')
xlsx_df = pd.read_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_带CID.xlsx')

# 根据 cid 合并 danmaku 列
merged_df = xlsx_df.merge(
    csv_df[['cid', 'danmaku']],  # 只保留需要的列
    on='cid',
    how='left'  # 保留 xlsx 中所有行，即使 csv 中没有匹配
)

# 保存为新文件（避免覆盖原文件）
merged_df.to_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_01.xlsx', index=False)

In [32]:
# 0. 载入自定义模型
sentiment_path = r'D:/NJU_WORKPIECE/Project/train.marshal'
sentiment.load(sentiment_path)

# 1. 读取 CSV 文件
csv_path = r'D:/NJU_WORKPIECE/Project/每周必看视频信息.csv'   # ← 换成你的实际路径
df = pd.read_csv(csv_path)

# 2. 定义打分函数（用你自己的模型）
def get_score(text: str) -> float:
    try:
        s = SnowNLP(str(text))
        return s.sentiments          # 0~1 之间，越接近 1 越正面
    except:
        return 0.5                   # 异常时给一个中性值

# 3. 应用打分并生成新列
df['score'] = df['title'].apply(get_score)

# 4. 保存结果（覆盖或另存）
output_path = r'D:/NJU_WORKPIECE/Project/标题评分_01.csv'  # ← 想覆盖就把路径设成原文件
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print('打分完成，已保存为：', output_path)

打分完成，已保存为： D:/NJU_WORKPIECE/Project/标题评分_01.csv


In [33]:
import pandas as pd

# 读取文件
csv_df = pd.read_csv('D:/NJU_WORKPIECE/Project/标题评分_01.csv')
xlsx_df = pd.read_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_带CID.xlsx')

# 根据 cid 合并 danmaku 列
merged_df = xlsx_df.merge(
    csv_df[['cid', 'score']],  # 只保留需要的列
    on='cid',
    how='left'  # 保留 xlsx 中所有行，即使 csv 中没有匹配
)

# 保存为新文件（避免覆盖原文件）
merged_df.to_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_02.xlsx', index=False)

In [1]:
import pandas as pd
import numpy as np

# 读取Excel文件
file_path = "D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_02.xlsx"
df = pd.read_excel(file_path)

# 定义修正函数
def correct_score(s, v, alpha):
    """
    根据公式计算修正后的得分
    s: SnowNLP均值
    v: title_score
    alpha: 修正力度系数
    """
    sign = np.sign(s - 0.5)
    corrected = 0.5 + (s - 0.5) * (1 - alpha * (v - 0.5) * sign)
    return corrected

# 计算三种alpha值下的修正得分
alphas = [0.5, 0.75, 1.0]
for alpha in alphas:
    df[f'修正后得分_α={alpha}'] = df.apply(lambda row: correct_score(row['SnowNLP均值'], row['title_score'], alpha), axis=1)

# 保存结果到新文件
output_path = "D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_03.xlsx"
df.to_excel(output_path, index=False)

print("修正完成！文件已保存为：每期弹幕_SnowNLP训练后评分_03.xlsx")
print("\n前5行数据预览：")
print(df[['期号', '文件名', 'SnowNLP均值', 'title_score', '修正后得分_α=0.5', '修正后得分_α=0.75', '修正后得分_α=1.0']].head())

# 显示一些统计信息
print("\n修正前后对比统计：")
print(f"原始SnowNLP均值 - 均值: {df['SnowNLP均值'].mean():.4f}, 标准差: {df['SnowNLP均值'].std():.4f}")
for alpha in alphas:
    col_name = f'修正后得分_α={alpha}'
    print(f"α={alpha}修正后 - 均值: {df[col_name].mean():.4f}, 标准差: {df[col_name].std():.4f}")

修正完成！文件已保存为：每期弹幕_SnowNLP训练后评分_03.xlsx

前5行数据预览：
   期号                   文件名  SnowNLP均值  title_score  修正后得分_α=0.5  \
0   1  第1期第7个视频82235716.txt      0.467     0.250663     0.471114   
1   1  第1期第8个视频82536177.txt      0.513     0.017560     0.516136   
2   1  第1期第2个视频81577663.txt      0.703     0.462417     0.706815   
3   1  第1期第1个视频82142406.txt      0.543     0.993110     0.532398   
4   1  第1期第5个视频82850118.txt      0.494     0.001113     0.495497   

   修正后得分_α=0.75  修正后得分_α=1.0  
0      0.473171     0.475228  
1      0.517704     0.519272  
2      0.708722     0.710629  
3      0.527097     0.521796  
4      0.496245     0.496993  

修正前后对比统计：
原始SnowNLP均值 - 均值: 0.5080, 标准差: 0.0766
α=0.5修正后 - 均值: 0.5163, 标准差: 0.0771
α=0.75修正后 - 均值: 0.5204, 标准差: 0.0779
α=1.0修正后 - 均值: 0.5245, 标准差: 0.0792


In [2]:
import pandas as pd

# 读取文件
csv_df = pd.read_csv('D:/NJU_WORKPIECE/Project/标题评分_01.csv')
xlsx_df = pd.read_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_03.xlsx')

# 根据 cid 合并 danmaku 列
merged_df = xlsx_df.merge(
    csv_df[['cid', 'danmaku']],  # 只保留需要的列
    on='cid',
    how='left'  # 保留 xlsx 中所有行，即使 csv 中没有匹配
)

# 保存为新文件（避免覆盖原文件）
merged_df.to_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04.xlsx', index=False)

In [3]:
import pandas as pd
import numpy as np

# 读取Excel文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04.xlsx'
df = pd.read_excel(file_path)

# 步骤1：按"期号"分组计算danmaku总数
danmaku_total_by_period = df.groupby('期号')['danmaku'].sum().reset_index()
danmaku_total_by_period.columns = ['期号', 'danmaku总数']

# 步骤2：将总数合并回原数据框
df = df.merge(danmaku_total_by_period, on='期号', how='left')

# 步骤3：计算每个元素对应的danmaku占比（权重）
df['权重'] = df['danmaku'] / df['danmaku总数']

# 步骤4：验证权重计算（可选）
# 检查每个期号的权重之和是否为1
# weight_check = df.groupby('期号')['权重'].sum()
# print("权重验证（每个期号权重和应为1）：")
# print(weight_check.head())

# 步骤5：保存结果到新文件
output_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04_加权.xlsx'
df.to_excel(output_path, index=False)

# 显示处理后的前几行数据和统计信息
print("处理后的数据预览：")
print(df[['期号', '文件名', 'danmaku', 'danmaku总数', '权重']].head(10))

print("\n各期danmaku总数统计：")
print(danmaku_total_by_period)

print(f"\n处理完成！文件已保存至：{output_path}")

处理后的数据预览：
   期号                   文件名  danmaku  danmaku总数        权重
0   1  第1期第7个视频82235716.txt    15795     302295  0.052250
1   1  第1期第8个视频82536177.txt   108893     302295  0.360221
2   1  第1期第2个视频81577663.txt    43882     302295  0.145163
3   1  第1期第1个视频82142406.txt    69870     302295  0.231132
4   1  第1期第5个视频82850118.txt     5865     302295  0.019402
5   1  第1期第6个视频82342112.txt    16584     302295  0.054860
6   1  第1期第4个视频81965902.txt    14326     302295  0.047391
7   1  第1期第9个视频78868295.txt    21927     302295  0.072535
8   1  第1期第3个视频82579802.txt     5153     302295  0.017046
9   2  第2期第9个视频83443650.txt    27167     222960  0.121847

各期danmaku总数统计：
      期号  danmaku总数
0      1     302295
1      2     222960
2      3     351171
3      4     163842
4      5     232900
..   ...        ...
343  345     888744
344  346     969238
345  347     792550
346  348     595330
347  349     520644

[348 rows x 2 columns]

处理完成！文件已保存至：D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04_加权.xlsx


In [4]:
import pandas as pd
import numpy as np

# 读取原始Excel文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04.xlsx'
df = pd.read_excel(file_path)

# 步骤1：按"期号"分组计算danmaku总数
danmaku_total_by_period = df.groupby('期号')['danmaku'].sum().reset_index()
danmaku_total_by_period.columns = ['期号', 'danmaku总数']

# 步骤2：将总数合并回原数据框
df = df.merge(danmaku_total_by_period, on='期号', how='left')

# 步骤3：计算每个元素对应的danmaku占比（权重）
df['权重'] = df['danmaku'] / df['danmaku总数']

# 步骤4：计算每个元素在不同α值下的加权得分
df['加权得分_α0.5'] = df['修正后得分_α=0.5'] * df['权重']
df['加权得分_α0.75'] = df['修正后得分_α=0.75'] * df['权重']
df['加权得分_α1.0'] = df['修正后得分_α=1.0'] * df['权重']

# 步骤5：按期号汇总加权得分
period_scores = df.groupby('期号').agg({
    '加权得分_α0.5': 'sum',
    '加权得分_α0.75': 'sum',
    '加权得分_α1.0': 'sum',
    'danmaku总数': 'first',
    '修正后得分_α=0.5': 'mean',
    '修正后得分_α=0.75': 'mean',
    '修正后得分_α=1.0': 'mean'
}).reset_index()

# 重命名列
period_scores.columns = ['期号', 
                         '每期加权得分_α0.5', '每期加权得分_α0.75', '每期加权得分_α1.0',
                         'danmaku总数', '平均修正得分_α0.5', '平均修正得分_α0.75', '平均修正得分_α1.0']

# 保存完整数据
output_path_full = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_04_完整加权.xlsx'
df.to_excel(output_path_full, index=False)

# 保存每期汇总得分
output_path_summary = 'D:/NJU_WORKPIECE/Project/每期弹幕_每期加权得分汇总.xlsx'
period_scores.to_excel(output_path_summary, index=False)

# 显示结果
print("每期加权得分汇总预览：")
print(period_scores.head(15))

print("\n不同α值加权得分对比：")
print(period_scores[['期号', '每期加权得分_α0.5', '每期加权得分_α0.75', '每期加权得分_α1.0']].head(10))

print(f"\n处理完成！")
print(f"完整数据已保存至：{output_path_full}")
print(f"每期汇总得分已保存至：{output_path_summary}")

每期加权得分汇总预览：
    期号  每期加权得分_α0.5  每期加权得分_α0.75  每期加权得分_α1.0  danmaku总数  平均修正得分_α0.5  \
0    1     0.547030      0.547312     0.547593     302295     0.534645   
1    2     0.496313      0.501919     0.507525     222960     0.511808   
2    3     0.510481      0.511410     0.512340     351171     0.527441   
3    4     0.527360      0.531640     0.535920     163842     0.531544   
4    5     0.538216      0.542921     0.547626     232900     0.527524   
5    6     0.526903      0.527825     0.528747     341783     0.553086   
6    7     0.553324      0.559785     0.566247     334673     0.511492   
7    8     0.459017      0.466558     0.474099     237614     0.519033   
8    9     0.504093      0.509672     0.515251     551786     0.522236   
9   10     0.530802      0.535709     0.540616     232742     0.551989   
10  11     0.542168      0.544314     0.546459     237765     0.543374   
11  12     0.531945      0.533756     0.535567     581242     0.518209   
12  13     0.529325      0

In [9]:
import pandas as pd
from pathlib import Path

file_path = r'D:/NJU_WORKPIECE/Project/每期弹幕_每期加权得分汇总.xlsx'
out_path  = r'D:/NJU_WORKPIECE/Project/月度数据.xlsx'

# 读取数据
df = pd.read_excel(file_path)

# 去掉可能的表头重复行
df = df[df['期号'] != '期号'].copy()

# 生成月份序号：每 4 期 = 1 个月，从 2019-03 开始
df['month_id'] = (df['期号'].astype(int) - 1) // 4
# 直接构造年月 Period，简单高效
df['year_month'] = pd.PeriodIndex(pd.to_datetime('2019-03-01') +
                                  pd.to_timedelta(df['month_id'] * 30, unit='D'), freq='M')

# 加权平均函数
def wmean(g, col):
    return (g[col] * g['danmaku总数']).sum() / g['danmaku总数'].sum()

# 按月聚合
monthly = (df.groupby('year_month')
             .apply(lambda g: pd.Series({
                 'danmaku总数': g['danmaku总数'].sum(),
                 '月度加权得分_α0.5':  wmean(g, '每期加权得分_α0.5'),
                 '月度加权得分_α0.75': wmean(g, '每期加权得分_α0.75'),
                 '月度加权得分_α1.0':  wmean(g, '每期加权得分_α1.0')
             }))
             .reset_index())

# 保存
monthly.to_excel(out_path, index=False)
print('已生成月度汇总文件：', out_path)

已生成月度汇总文件： D:/NJU_WORKPIECE/Project/月度数据.xlsx


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\2534940718.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [5]:
file_path = r'D:/NJU_WORKPIECE/Project/每周必看视频信息.csv'
df = pd.read_csv(file_path)
df.head(30)

,number,title,cid,pubdate,name,tname,desc,view,danmaku,reply,favorite,coin,share,now_rank,his_rank,like,dislike,tnamev2,pid_name_v2,rcmd_reason
0,1,这集vlog我们拍了十年，致最美好的青春,82142406,1553152210,AresserA木恩木,出行,我在18岁认识了你，\n然后我们开始了长达8年的异地恋，\n2019年03月16日\n我们在...,5348902,69870,22524,238393,553343,212641,0,1,508072,0,其他vlog,vlog,暴风流泪推荐！今天也是为别人的神仙爱情流泪的一天！
1,1,【性转版】回家的诱惑,81577663,1552823023,兰彻lancche,影视剪辑,性转版回家的诱惑\n都市男人拯救幸福情仇大戏\n认真你就输了23333,4894417,43882,12878,148150,216491,160342,0,1,251764,0,影视剪辑,影视,无论是创意、配音还是剪辑，全都是超高质量！
2,1,哆啦A梦结局背后的秘密？从未播出的黑历史？真相出人意料,82579802,1553396443,瓶子君152,动漫杂谈,微博：@瓶子君152\n关注关注关注关注关注关注关注关注关注关注关注关注\n三连三连三连三连...,2649566,5153,3448,52551,134756,4301,0,6,161977,0,动漫评论,二次元,NaN
3,1,在中国挑战通宵卖烧烤！为啥美国没有这个？,81965902,1553050747,我是郭杰瑞,美食侦探,在美国我们很少有深夜美食，但是在中国却很多，那么，通宵卖烧烤是怎样的一种生活呢？\n\n关注...,2645843,14326,3526,5160,18926,2078,0,5,77502,0,美食记录,美食,NaN
4,1,【纯黑】《鬼泣5》一周目无伤S评价攻略解说 第七期,82850118,1553501956,-纯黑-,单机游戏,一周目最高难度，不是游戏最高难度。剧情杀以外无伤。\n微博：@纯黑GK \n直播间：www....,1529737,5865,1965,19085,75727,1069,0,4,65722,0,单机主机类游戏,游戏,NaN
5,1,化身修罗只为守护你！此刻世界将为之颤抖！,82342112,1553256958,烟季,MAD·AMV,"BGM：「Tommee Profitt,Jung Youth,Fleurie - In th...",7252131,16584,6302,318739,417867,31787,0,1,410137,0,动漫剪辑,二次元,NaN
6,1,破案大师老番茄,82235716,1553227211,老番茄,单机游戏,游戏：Return of the Obra Dinn,6651912,15795,4201,27169,51791,3052,0,4,141109,0,单机主机类游戏,游戏,NaN
7,1,[LPL春季赛] 3月23日 RNG vs IG,82536177,1553347231,哔哩哔哩英雄联盟赛事,电子竞技,[LPL春季赛] 3月23日 RNG vs IG,4057575,108893,46348,21383,26046,5877,0,9,41035,0,MOBA游戏,游戏,NaN
8,1,the real suger baby已知最高画质,78868295,1551437298,一嗷垚,音乐综合,转自油管,29820209,21927,19540,677419,242050,105026,0,65,783888,0,颜值·网红舞,舞蹈,NaN
9,2,女主人扮恐龙把猫吓的到处飞，直接六亲不认，一脱衣服，猫：妈？,83471931,1553851043,花花与三猫CatLive,喵星人,调皮铲屎官又耍花招\n装扮成恐龙吓唬猫咪\n不出意料！\n果然猫咪被吓得撒腿就跑！\n猫：这...,2747155,12844,4206,25310,89631,12197,0,11,121819,0,猫,动物,调皮铲屎官又耍花招，装扮成恐龙吓唬猫咪！还是亲妈最会玩_(:3」∠)_


In [10]:
import pandas as pd

# 读取文件
csv_df = pd.read_csv('D:/NJU_WORKPIECE/Project/每周必看视频信息.csv')
xlsx_df = pd.read_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_带CID.xlsx')

# 根据 cid 合并 danmaku 列
merged_df = xlsx_df.merge(
    csv_df[['cid', 'danmaku', 'tname']],  # 只保留需要的列
    on='cid',
    how='left'  # 保留 xlsx 中所有行，即使 csv 中没有匹配
)

# 保存为新文件（避免覆盖原文件）
merged_df.to_excel('D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001.xlsx', index=False)

In [11]:
import pandas as pd

# 读取文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001.xlsx'
df = pd.read_excel(file_path)

# 按tname分组，对SnowNLP均值做去中心化处理（即减去该组的均值）
df['新评分'] = df.groupby('tname')['SnowNLP均值'].transform(lambda x: x - x.mean())

# 保存结果到新文件（可选）
output_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_去中心化.xlsx'
df.to_excel(output_path, index=False)

# 显示前几行结果
print(df[['期号', 'tname', 'SnowNLP均值', '新评分']].head())

   期号 tname  SnowNLP均值       新评分
0   1  单机游戏      0.467 -0.039856
1   1  电子竞技      0.513 -0.001770
2   1  影视剪辑      0.703  0.179698
3   1    出行      0.543  0.051133
4   1  单机游戏      0.494 -0.012856


In [12]:
import pandas as pd

# 读取 Excel 文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_去中心化.xlsx'
df = pd.read_excel(file_path)

# 按“期号”分组，计算“新评分”的加权平均值，权重为“danmaku”
weighted_avg = df.groupby('期号').apply(
    lambda x: (x['新评分'] * x['danmaku']).sum() / x['danmaku'].sum()
).reset_index(name='加权平均评分')

# 输出结果
print(weighted_avg)

# 可选：保存结果到新的 Excel 文件
output_path = 'D:/NJU_WORKPIECE/Project/每期加权平均评分.xlsx'
weighted_avg.to_excel(output_path, index=False)

      期号    加权平均评分
0      1  0.036557
1      2 -0.018363
2      3  0.004090
3      4  0.004296
4      5  0.015812
..   ...       ...
343  345  0.022126
344  346  0.007284
345  347  0.017868
346  348 -0.050866
347  349 -0.048080

[348 rows x 2 columns]


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\2998181303.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_avg = df.groupby('期号').apply(


In [13]:
import pandas as pd
import numpy as np

# 读取文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001.xlsx'
df = pd.read_excel(file_path)

# 根据tname列进行分组去中心化处理
def demean_and_normalize(group):
    # 去中心化：减去组内均值
    demeaned = group['SnowNLP均值'] - group['SnowNLP均值'].mean()
    
    # 归一化到0-1范围
    min_val = demeaned.min()
    max_val = demeaned.max()
    
    # 如果最大值等于最小值（所有值相同），则直接返回0.5
    if max_val == min_val:
        normalized = pd.Series(0.5, index=group.index)
    else:
        # 使用min-max归一化到0-1范围
        normalized = (demeaned - min_val) / (max_val - min_val)
    
    return normalized

# 应用函数并创建新列
df['新评分'] = df.groupby('tname').apply(demean_and_normalize).reset_index(level=0, drop=True)

# 保存结果到新文件（保留原始数据并添加新列）
output_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_处理后.xlsx'
df.to_excel(output_path, index=False)

print(f"处理完成！结果已保存到: {output_path}")
print(f"处理后的数据统计：")
print(f"总行数: {len(df)}")
print(f"新评分列的统计信息：")
print(df['新评分'].describe())

# 显示前几行数据作为示例
print("\n前5行数据示例：")
print(df[['tname', 'SnowNLP均值', '新评分']].head())

C:\Users\18414\AppData\Local\Temp\ipykernel_13836\1620753078.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['新评分'] = df.groupby('tname').apply(demean_and_normalize).reset_index(level=0, drop=True)


处理完成！结果已保存到: D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_处理后.xlsx
处理后的数据统计：
总行数: 12816
新评分列的统计信息：
count    12816.000000
mean         0.448247
std          0.186125
min          0.000000
25%          0.320604
50%          0.443599
75%          0.567956
max          1.000000
Name: 新评分, dtype: float64

前5行数据示例：
  tname  SnowNLP均值       新评分
0  单机游戏      0.467  0.537748
1  电子竞技      0.513  0.489637
2  影视剪辑      0.703  0.830735
3    出行      0.543  0.418891
4  单机游戏      0.494  0.573510


In [15]:
import pandas as pd

# 读取 Excel 文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_处理后.xlsx'
df = pd.read_excel(file_path)

# 按“期号”分组，计算加权平均评分和danmaku总和
result = df.groupby('期号').agg(
    加权平均评分=('新评分', lambda x: (x * df.loc[x.index, 'danmaku']).sum() / df.loc[x.index, 'danmaku'].sum()),
    danmaku总和=('danmaku', 'sum')
).reset_index()

# 输出结果
print(result)

# 保存结果到新的 Excel 文件
output_path = 'D:/NJU_WORKPIECE/Project/每期加权平均评分及danmaku总和.xlsx'
result.to_excel(output_path, index=False)

      期号    加权平均评分  danmaku总和
0      1  0.542316     302295
1      2  0.425170     222960
2      3  0.500493     351171
3      4  0.480141     163842
4      5  0.580211     232900
..   ...       ...        ...
343  345  0.516807     888744
344  346  0.469735     969238
345  347  0.517654     792550
346  348  0.327716     595330
347  349  0.294198     520644

[348 rows x 3 columns]


In [14]:
import pandas as pd

# 读取 Excel 文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_001_处理后.xlsx'
df = pd.read_excel(file_path)

# 按“期号”分组，计算“新评分”的加权平均值，权重为“danmaku”
weighted_avg = df.groupby('期号').apply(
    lambda x: (x['新评分'] * x['danmaku']).sum() / x['danmaku'].sum()
).reset_index(name='加权平均评分')

# 输出结果
print(weighted_avg)

# 可选：保存结果到新的 Excel 文件
output_path = 'D:/NJU_WORKPIECE/Project/每期加权平均评分（新）.xlsx'
weighted_avg.to_excel(output_path, index=False)

      期号    加权平均评分
0      1  0.542316
1      2  0.425170
2      3  0.500493
3      4  0.480141
4      5  0.580211
..   ...       ...
343  345  0.516807
344  346  0.469735
345  347  0.517654
346  348  0.327716
347  349  0.294198

[348 rows x 2 columns]


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\4049039796.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_avg = df.groupby('期号').apply(


In [18]:
import pandas as pd
from pathlib import Path

file_path = r'D:/NJU_WORKPIECE/Project/每期加权平均评分及danmaku总和.xlsx'
out_path  = r'D:/NJU_WORKPIECE/Project/月度数据02.xlsx'

# 读取数据
df = pd.read_excel(file_path)

# 去掉可能的表头重复行
df = df[df['期号'] != '期号'].copy()

# 生成月份序号：每 4 期 = 1 个月，从 2019-03 开始
df['month_id'] = (df['期号'].astype(int) - 1) // 4
# 直接构造年月 Period，简单高效
df['year_month'] = pd.PeriodIndex(pd.to_datetime('2019-03-01') +
                                  pd.to_timedelta(df['month_id'] * 30, unit='D'), freq='M')

# 加权平均函数
def wmean(g, col):
    return (g[col] * g['danmaku总和']).sum() / g['danmaku总和'].sum()

# 按月聚合
monthly = (df.groupby('year_month')
             .apply(lambda g: pd.Series({
                 'danmaku总数': g['danmaku总和'].sum(),
                 '月度加权评分':  wmean(g, '加权平均评分'),
             }))
             .reset_index())

# 保存
monthly.to_excel(out_path, index=False)
print('已生成月度汇总文件：', out_path)

已生成月度汇总文件： D:/NJU_WORKPIECE/Project/月度数据02.xlsx


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\2077170095.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [19]:
import pandas as pd

# 读取 Excel 文件
file_path = 'D:/NJU_WORKPIECE/Project/每期弹幕_SnowNLP训练后评分_01.xlsx'
df = pd.read_excel(file_path)

# 计算每一期的加权平均 SnowNLP 均值
def weighted_average(group):
    weights = group['danmaku']
    values = group['SnowNLP均值']
    return (values * weights).sum() / weights.sum()

# 按“期号”分组并计算加权平均
weighted_avg = df.groupby('期号').apply(weighted_average).reset_index()
weighted_avg.columns = ['期号', '加权平均SnowNLP均值']

# 计算每一期的总 danmaku 数量
danmaku_sum = df.groupby('期号')['danmaku'].sum().reset_index()
danmaku_sum.columns = ['期号', '总danmaku']

# 合并结果
result = pd.merge(weighted_avg, danmaku_sum, on='期号')

# 保存结果到新的 Excel 文件
output_path = 'D:/NJU_WORKPIECE/Project/每期加权平均评分001.xlsx'
result.to_excel(output_path, index=False)

print("处理完成，结果已保存到：", output_path)

处理完成，结果已保存到： D:/NJU_WORKPIECE/Project/每期加权平均评分001.xlsx


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\1881054492.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_avg = df.groupby('期号').apply(weighted_average).reset_index()


In [20]:
import pandas as pd
from pathlib import Path

file_path = r'D:/NJU_WORKPIECE/Project/每期加权平均评分001.xlsx'
out_path  = r'D:/NJU_WORKPIECE/Project/月度数据03.xlsx'

# 读取数据
df = pd.read_excel(file_path)

# 去掉可能的表头重复行
df = df[df['期号'] != '期号'].copy()

# 生成月份序号：每 4 期 = 1 个月，从 2019-03 开始
df['month_id'] = (df['期号'].astype(int) - 1) // 4
# 直接构造年月 Period，简单高效
df['year_month'] = pd.PeriodIndex(pd.to_datetime('2019-03-01') +
                                  pd.to_timedelta(df['month_id'] * 30, unit='D'), freq='M')

# 加权平均函数
def wmean(g, col):
    return (g[col] * g['总danmaku']).sum() / g['总danmaku'].sum()

# 按月聚合
monthly = (df.groupby('year_month')
             .apply(lambda g: pd.Series({
                 'danmaku总数': g['总danmaku'].sum(),
                 '月度加权评分':  wmean(g, '加权平均SnowNLP均值'),
             }))
             .reset_index())

# 保存
monthly.to_excel(out_path, index=False)
print('已生成月度汇总文件：', out_path)

已生成月度汇总文件： D:/NJU_WORKPIECE/Project/月度数据03.xlsx


C:\Users\18414\AppData\Local\Temp\ipykernel_13836\2045096553.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [6]:
import pandas as pd

# 1. 读取文件
file_path = 'D:/NJU_WORKPIECE/经济指标&情绪评分.xlsx'
df = pd.read_excel(file_path)

# 2. 对“社会消费品零售总额当期值(亿元)”列进行线性插值
df['社会消费品零售总额当期值(亿元)'] = df['社会消费品零售总额当期值(亿元)'].interpolate(method='linear')

# 3. 保存为新文件（覆盖原文件请把文件名改成原文件名）
output_path = 'D:/NJU_WORKPIECE/经济指标_线性插值填充.xlsx'
df.to_excel(output_path, index=False)

print(f'已保存插值后的文件：{output_path}')

已保存插值后的文件：D:/NJU_WORKPIECE/经济指标_线性插值填充.xlsx


In [ ]:
总体是这个样子的：
1.对于中心化的模型：训练后snownlp、nb不太好，snownlp>hh（情绪有能力预测社零）且显著性都还可以 svm则是相反的（情绪可以被社零有效预测）；
2.对于标题修正的模型：训练后snownlp、nb同样不太行，snownlp（情绪有能力预测社零）且显著性不错    svm则是相反（情绪可以被社零）
3.对于原始得分的：snownlp原始得分、nb出现伪回归；训练snownlp原始得分和hh不太好(显著性不强)；
所以：我们的所有评分中：snownlp中心化、snownlp标题修正、hh中心化、svm中心化、svm标题修正都是可行的
ps:svm出现“反常”的原因可能是：预测情绪（SVM 原始得分与 SnowNLP/hh 的量纲、分布、对语义敏感度不同。SVM 得分可能更多捕捉“事后吐槽”（买完再评论），而非“事前预期”（想买前先讨论）。结果就变成：当官方社零数据公布或实际消费升温后，网友才在弹幕里滞后地表达“买了”“涨价了”，于是出现社零→情绪显著。）